# Build one ore-processing learning example

Level: 1. Prerequisites: None; begin here. Reading time: 20 minutes.

## Objectives

Use OreFlow's own feature and target functions on one authored circuit. Identify what the surrogate learns and why engine-generated rows are not plant observations.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Configure imports and record versions. This makes the method run reproducible.

In [1]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the local pipeline from the repository-relative layout.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Data

Owner: OreFlow authors; authored circuits and generated learning results in `data/derived/learning.json`, MIT project licence. The training record uses twelve constructed circuits, not plant historian data. Repository: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

The supervised record maps physical ore and operating descriptors $x$ to recovery, $\log_{10}(G_c/G_f)$ and total specific energy. Case identity is excluded to prevent a shortcut. `features` and `targets` in `pipeline/methods/learning.py` create the production representation.

Ejecuta un caso de autoría propia y llama las mismas transformaciones de características y objetivos usadas para construir el diseño de aprendizaje.

In [2]:
from pipeline.cases.catalog import CASES
from pipeline.engine.circuit import simulate
from pipeline.methods.learning import FEATURES, TARGETS, features, targets

# Make one labeled example with the product's process solver and transformations.
case = next(item for item in CASES if item.id == 'copper_porphyry_soft')
result = simulate(case.ore, case.plant, case.nominal)
x = features(case, case.nominal, {'liberation_size': 1.0, 'floatability': 1.0})
y = targets(result.metrics)
print('features:', len(FEATURES), '| targets:', len(TARGETS))
print('target labels:', dict(zip(TARGETS, y)))

features: 22 | targets: 3
target labels: {'recovery_pct': 92.21261928007924, 'log_upgrade': 1.5526299674405057, 'specific_energy_total_kwh_t': 10.394628844259875}


Inspecciona el vector con los nombres y unidades del pipeline. Esto recibe el modelo supervisado; no se incluye el ID de caso.

In [3]:
# Inspecciona el vector con los nombres y unidades del pipeline.
display(pd.DataFrame({'feature': FEATURES, 'value': x}))
display(pd.Series(dict(zip(TARGETS, y)), name='engine-generated target').to_frame())

,feature,value
0,log_payable_fraction,-2.130768
1,specific_throughput_t_h_mw,99.037139
2,target_p80_um,150.000000
3,circulating_load,2.500000
4,water_m3_t,2.100000
5,crusher_css_mm,8.000000
6,work_index_kwh_t,11.000000
7,carrier_liberation_um,110.000000
8,grind_to_liberation,1.363636
9,carrier_composite_content,0.350000


,engine-generated target
recovery_pct,92.212619
log_upgrade,1.552630
specific_energy_total_kwh_t,10.394629


Compara los objetivos con las métricas del motor resuelto. Confirma procedencia, pero no constituye validación independiente.

In [4]:
# Compara los objetivos con las métricas del motor resuelto.
print('engine recovery (%):', result.metrics['recovery_pct'])
print('engine concentrate grade:', result.metrics['concentrate_grade'])
print('engine specific energy (kWh/t):', result.metrics['specific_energy_total_kwh_t'])

engine recovery (%): 92.21261928007924
engine concentrate grade: 26.415673447496314
engine specific energy (kWh/t): 10.394628844259875


## Results

The row shows the exact input and labels for one deterministic engine run. Because the engine creates the label, this example teaches the data contract; it cannot establish surrogate accuracy on a real plant.

## Summary

Inspect feature construction and target provenance before reading model scores. A surrogate that reproduces the engine learns that engine's assumptions.

## References

- https://doi.org/10.1023/A:1010933404324
- https://doi.org/10.1023/A:1010933404324